In [5]:
!pip install --upgrade pip setuptools wheel -q
!pip install --upgrade cmake -q
!pip install scs --prefer-binary -q
!pip install cvxpy --prefer-binary -q
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
!pip install catboost


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Program Files\Python314\python.exe -m pip install --upgrade pip setuptools wheel -q

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program F

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [6]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from optbinning import BinningProcess
import shutil
from warnings import simplefilter
simplefilter(action = "ignore") #, category = FutureWarning

pd.set_option('display.max_rows', 500)
from sklearn.preprocessing import LabelEncoder

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple
#import plotly.graph_objects as go
#from plotly.subplots import make_subplots
import os
#import plotly.express as px

pd.set_option('display.float_format', '{:.2f}'.format)

In [8]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [10]:
%%time
query = """

WITH pd AS (
    SELECT DISTINCT
        -- Identificadores
        a.key_value,
        a.cod_cli,
        date_format(
            date_parse(CAST(a.cod_mes AS varchar), '%Y%m') - interval '1' month,
            '%Y%m'
        ) AS codmes_lag1,
        CAST(a.cod_mes AS INTEGER) AS cod_mes,

        -- Fechas
        TRY_CAST(a.fec_constitucion AS DATE) AS fec_constitucion,

        -- Monetarios
        TRY_CAST(a.mto_pas_soles AS DOUBLE) AS mto_pas_soles,
        TRY_CAST(a.imp_trx_abonosefect_6m AS DOUBLE) AS imp_trx_abonosefect_6m,
        TRY_CAST(a.imp_trx_cargosefe_6m AS DOUBLE) AS imp_trx_cargosefe_6m,
        TRY_CAST(a.avg_trx_cargostot_3m AS DOUBLE) AS avg_trx_cargostot_3m,
    TRY_CAST(a.max_trx_abonos_3m AS DOUBLE) AS max_trx_abonos_3m,

        -- Cantidades
        TRY_CAST(a.cnt_trx_cargostot_3m AS INTEGER) AS cnt_trx_cargostot_3m,

        -- Promedios / ratios
        TRY_CAST(a.cnt_trx_abonospromtot_3m AS DOUBLE) AS cnt_trx_abonospromtot_3m,
        TRY_CAST(a.rat_trx_abonosefectot_1m AS DOUBLE) AS rat_trx_abonosefectot_1m,
        TRY_CAST(a.rat_trx_abonosefectot_3m AS DOUBLE) AS rat_trx_abonosefectot_3m,
        TRY_CAST(a.rat_trx_abonosefectot_9m AS DOUBLE) AS rat_trx_abonosefectot_9m,
        TRY_CAST(a.rat_mntcrgsefetot_1m AS DOUBLE) AS rat_mntcrgsefetot_1m,

        -- Demográficas / antigüedad
        TRY_CAST(a.num_edad_constitucion AS INTEGER) AS num_edad_constitucion,
    TRY_CAST(a.num_antiguedad AS INTEGER) AS num_antiguedad,

        -- Riesgo
        TRY_CAST(a.desc_nivel_rsg_lsb_tot AS DOUBLE) AS desc_nivel_rsg_lsb_tot,

        -- Actividad mensual
        TRY_CAST(a.cnt_meses_siningresos_12m AS INTEGER) AS cnt_meses_siningresos_12m,
        TRY_CAST(a.cnt_meses_sinegresos_12m AS INTEGER) AS cnt_meses_sinegresos_12m,

        -- Ubicación / segmentación
        a.desc_provincia,
        a.desc_departamento,
        a.cod_ubigeo_cd,
        a.cod_sectorista_id,
        a.cod_ciiu_v4,

        -- Flags (string/bool → 0/1)
        CAST(a.flg_casos_hist AS INTEGER) AS flg_casos_hist,
        CAST(a.flg_vrcn_abonos_5m_1m AS INTEGER) AS flg_vrcn_abonos_5m_1m,
        CAST(a.flg_vrcn_efe_cargos_5m_1m AS INTEGER) AS flg_vrcn_efe_cargos_5m_1m,

        -- Conteos
        a.cnt_ro_debajo_umbral,
        -- Perfil económico
        a.mto_fact_declarado_sunat,
        TRY_CAST(a.avg_cp_men_ing_12m AS DOUBLE) AS avg_cp_men_ing_12m,
        TRY_CAST(a.avg_cpmenegr_12m AS DOUBLE) AS avg_cpmenegr_12m,
        TRY_CAST(a.max_mto_cpmening_12m AS DOUBLE) AS max_mto_cpmening_12m,
        TRY_CAST(a.max_mto_cpegrmen_12m AS DOUBLE) AS max_mto_cpegrmen_12m,

        -- Exterior
        a.flg_al_ext_12m,
        a.flg_del_ext_12m,
        TRY_CAST(a.cnt_trx_sinenv_alext_12m AS INTEGER) AS cnt_trx_sinenv_alext_12m,
        TRY_CAST(a.cnt_trx_al_ext_1000_12m AS INTEGER) AS cnt_trx_al_ext_1000_12m,
        TRY_CAST(a.mto_al_ext_12m AS DOUBLE) AS mto_al_ext_12m,
        TRY_CAST(a.mto_del_ext_12m AS DOUBLE) AS mto_del_ext_12m,

        -- Reputacional / antecedentes
        a.flg_pep,
        a.cod_rsg_pep,
        a.flg_activo_pep,
        TRY_CAST(a.cnt_noticias AS INTEGER) AS cnt_noticias,
        a.flg_ros_12m,
        a.flg_alerta_12m,
        TRY_CAST(a.cnt_alerta_hist AS INTEGER) AS cnt_alerta_hist,
        TRY_CAST(a.cnt_ros_hist AS INTEGER) AS cnt_ros_hist,

        -- KYC
        a.flg_kyc_12m,
        a.flg_kyc_hist,
        TRY_CAST(a.cnt_kyc_hist AS INTEGER) AS cnt_kyc_hist,
        -- ======================================================
        -- 🔹 ACELERACIÓN / CAMBIO DE COMPORTAMIENTO
        -- ======================================================
        TRY_CAST(a.imp_trx_abonostot_1m AS DOUBLE)
            / NULLIF(TRY_CAST(a.avg_trx_abonostot_6m AS DOUBLE), 0)
            AS rat_abonos_1m_vs_6m,

        TRY_CAST(a.imp_trx_cargostot_1m AS DOUBLE)
            / NULLIF(TRY_CAST(a.avg_trx_cargostot_6m AS DOUBLE), 0)
            AS ratio_cargos_1m_vs_6m,


        -- ======================================================
        -- 🔹 CONCENTRACIÓN EN CONTRAPARTE
        -- ======================================================
        TRY_CAST(a.avg_cpmenegr_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_cargostot_6m AS DOUBLE), 0)
            AS share_cp_egresos,

        TRY_CAST(a.avg_cp_men_ing_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_6m AS DOUBLE), 0)
            AS share_cp_ingresos,


        -- ======================================================
        -- 🔹 NORMALIZACIÓN DE RIESGO
        -- ======================================================
        TRY_CAST(a.cnt_ros_hist AS DOUBLE)
            / NULLIF(TRY_CAST(a.cnt_trx_cargostot_3m AS DOUBLE), 0)
            AS rat_cntros_x_cnttrxegr_3m,

        TRY_CAST(a.cnt_alerta_hist AS DOUBLE)
            / NULLIF(TRY_CAST(a.num_antiguedad AS DOUBLE), 0)
            AS alertas_por_antiguedad,


        -- ======================================================
        -- 🔹 COHERENCIA ECONÓMICA
        -- ======================================================
        TRY_CAST(a.imp_trx_abonostot_6m AS DOUBLE)
            / NULLIF(TRY_CAST(a.mto_fact_declarado_sunat AS DOUBLE), 0)
            AS rat_ing_tot_x_factura_6m,

        TRY_CAST(a.mto_pas_soles AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_6m AS DOUBLE), 0)
            AS rat_pastot_x_ingtot_6m,



        -- ======================================================
        -- 🔹 EXPOSICIÓN AL EXTERIOR (PROPORCIONES)
        -- ======================================================
        TRY_CAST(a.mto_al_ext_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_12m AS DOUBLE), 0)
            AS ratio_egresos_exterior,

        TRY_CAST(a.mto_del_ext_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_12m AS DOUBLE), 0)
            AS rat_ing_ext_x_ing_tot_12m,

        -- ======================================================
        -- 🔹 COHERENCIA PEP / LSB
        -- ======================================================
        TRY_CAST(a.cod_rsg_pep AS DOUBLE)
            - TRY_CAST(a.desc_nivel_rsg_lsb_tot AS DOUBLE)
            AS gap_riesgo_pep_lsb

    FROM d_perm_aws.t_agg_alertas_plaft a
        WHERE a.cod_mes BETWEEN '202501' AND '202604'
            AND a.desc_subsegmento = 'BPE'
),

target AS (
    SELECT 
        codunico,
        periodo_alerta,
        tipo_alerta_n2,
        trx_riesgo_cliente,
        MAX(calificacion_monitoreo) AS flg_alerta
    FROM e_perm_aws.t_alertas_plaft
    GROUP BY codunico, periodo_alerta, tipo_alerta_n2,trx_riesgo_cliente
)

SELECT 
    a.*,
    b.tipo_alerta_n2,
    b.trx_riesgo_cliente, 
    CASE 
        WHEN b.flg_alerta = '1' THEN 1 
        ELSE 0 
    END AS target_m
FROM pd a
LEFT JOIN target b
    ON a.cod_cli = b.codunico
    AND cast(a.cod_mes as varchar) = b.periodo_alerta
--   AND codmes_lag1 = c.periodo_alerta
;"""
df = athena_query(query, database='disc_comercial')
df.head()

CPU times: total: 51.9 s
Wall time: 5min 42s


,key_value,cod_cli,codmes_lag1,cod_mes,fec_constitucion,mto_pas_soles,imp_trx_abonosefect_6m,imp_trx_cargosefe_6m,avg_trx_cargostot_3m,max_trx_abonos_3m,...,rat_cntros_x_cnttrxegr_3m,alertas_por_antiguedad,rat_ing_tot_x_factura_6m,rat_pastot_x_ingtot_6m,ratio_egresos_exterior,rat_ing_ext_x_ing_tot_12m,gap_riesgo_pep_lsb,tipo_alerta_n2,trx_riesgo_cliente,target_m
0,CFBB0B2F5FD6513C86D87E443AE6AB77E602A9E2838E92...,0019624887,202507,202508,None,3199.80,0.00,0.00,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,0
1,5EDCFFE54DFF78B5822DC3A6D788AB6492503FEB107E61...,0021415266,202507,202508,2024-11-13,249919.55,0.00,657954.28,7.00,143272.41,...,0.00,NaN,NaN,0.30,NaN,1.00,NaN,<NA>,<NA>,0
2,726792FA75C0F81CE4B29414E4E356015115AA8A1314F8...,0020771814,202507,202508,2022-10-05,6083.76,0.00,0.00,0.00,780.03,...,NaN,NaN,NaN,1.60,NaN,NaN,NaN,<NA>,<NA>,0
3,66AE4C9494ACC2F30258D65887469A6EDC8D9F0CF878DB...,0017575538,202502,202503,2016-08-23,0.00,0.00,0.00,0.00,0.00,...,NaN,NaN,0.00,0.00,NaN,NaN,NaN,<NA>,<NA>,0
4,39E49C2981091DF00BDEDB978F1F1AC3732660799CA1E7...,0021293926,202503,202504,2024-09-25,112915.83,18494.90,65328.20,1.00,63405.26,...,NaN,NaN,NaN,0.34,NaN,NaN,NaN,<NA>,<NA>,0


In [11]:

# ============================================================
# 🔍 VALIDACIÓN DE VARIABLES POST-QUERY
# ============================================================
print("=" * 80)
print("📊 RESUMEN DE DATAFRAME DEL QUERY SQL")
print("=" * 80)

print(f"\n✓ Registros totales: {len(df):,}")
print(f"✓ Columnas totales: {len(df.columns)}")

print("\n📋 LISTADO DE COLUMNAS:")
print("-" * 80)
for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    non_null = df[col].notna().sum()
    null_pct = (df[col].isna().sum() / len(df) * 100)
    print(f"{i:3d}. {col:40s} | Tipo: {str(dtype):20s} | Nulos: {null_pct:5.2f}%")

print("\n" + "=" * 80)
print("📌 TIPOS DE DATOS DETECTADOS:")
print("=" * 80)
print(df.dtypes)

print("\n" + "=" * 80)
print("🔗 VALIDACIÓN DE KEY VARIABLES:")
print("=" * 80)
key_vars = ['key_value', 'cod_cli', 'cod_mes', 'target_m', 'tipo_alerta_n2', 
            'mto_pas_soles', 'cnt_trx_cargostot_3m', 'num_antiguedad']
for var in key_vars:
    if var in df.columns:
        print(f"✓ {var:30s} está presente")
    else:
        print(f"✗ {var:30s} FALTA")

print("\nℹ️ Primeras 5 filas:")
print(df.head())


📊 RESUMEN DE DATAFRAME DEL QUERY SQL

✓ Registros totales: 2,619,703
✓ Columnas totales: 66

📋 LISTADO DE COLUMNAS:
--------------------------------------------------------------------------------
  1. key_value                                | Tipo: string               | Nulos:  0.00%
  2. cod_cli                                  | Tipo: string               | Nulos:  0.00%
  3. codmes_lag1                              | Tipo: string               | Nulos:  0.00%
  4. cod_mes                                  | Tipo: Int32                | Nulos:  0.00%
  5. fec_constitucion                         | Tipo: object               | Nulos:  3.02%
  6. mto_pas_soles                            | Tipo: float64              | Nulos:  0.00%
  7. imp_trx_abonosefect_6m                   | Tipo: float64              | Nulos: 17.09%
  8. imp_trx_cargosefe_6m                     | Tipo: float64              | Nulos: 17.09%
  9. avg_trx_cargostot_3m                     | Tipo: float64              

In [12]:
df_1=df.copy()

In [13]:
# Renombrar y reordenar columnas
df_1 = df_1.rename(columns={'target_m': 'target'})
df_1 = df_1[['target'] + [c for c in df_1.columns if c != 'target']]

In [14]:
df_1 = df_1[['target'] + [c for c in df_1.columns if c != 'target']]
df_1 = df_1.drop_duplicates(subset=['key_value', 'cod_mes'], keep='first')

In [15]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Copiar DF original
# ==========================================
df_2 = df_1.copy()

# ==========================================
# 3. Columnas Int32 → rellenar NA → convertir a int64
# ==========================================
int32_cols = df_2.select_dtypes(include=["Int32"]).columns

df_2[int32_cols] = df_2[int32_cols].fillna(0).astype("int64")

# ==========================================
# 4. Columnas float → rellenar NA con 0
# ==========================================
float_cols = df_2.select_dtypes(include=["float64", "Float64"]).columns

df_2[float_cols] = df_2[float_cols].fillna(0)

# ==========================================
# 5. Columnas boolean → rellenar NA con False
# ==========================================
bool_cols = df_2.select_dtypes(include=["boolean"]).columns

df_2[bool_cols] = df_2[bool_cols].fillna(False)

# ==========================================
# 6. Columnas categóricas (strings) → NA = "SIN_INFO"
# ==========================================
cat_cols = df_2.select_dtypes(include=["object", "string"]).columns

df_2[cat_cols] = df_2[cat_cols].fillna("SIN_INFO")


In [16]:
df_2[['cod_mes', 'target']].value_counts()

cod_mes  target
202604   0         172271
202512   0         171315
202601   0         170951
202603   0         170300
202602   0         168998
202511   0         166252
202510   0         164651
202507   0         163470
202506   0         162766
202509   0         162739
202508   0         161296
202505   0         158189
202504   0         157310
202503   0         155867
202502   0         155562
202501   0         155135
202604   1            234
202602   1            150
202505   1            147
202502   1            145
202503   1            137
202603   1            126
202501   1            125
202510   1            123
202601   1            119
202508   1            119
202506   1            117
202504   1            109
202509   1             99
202507   1             97
202511   1             65
202512   1             16
Name: count, dtype: int64

In [17]:
import pandas as pd
from sklearn.utils import resample

# ===========================
# 1️⃣ Separar antiguos y recientes
# ===========================
df_antiguos = df_2[df_2['cod_mes'] <= 202507].copy()
df_recientes = df_2[df_2['cod_mes'].between(202508, 202604)].copy()
#df_recientes = df_2[df_2['cod_mes'] >= 202508].copy()  # Test completo

# ===========================
# 2️⃣ Separar clases en antiguos
# ===========================
df_antiguos_con_alerta = df_antiguos[df_antiguos['tipo_alerta_n2'] != "0"]  # Con alerta
df_antiguos_sin_alerta = df_antiguos[df_antiguos['tipo_alerta_n2'] == "0"]  # Sin alerta

# ===========================
# 3️⃣ Filtrar clases target == 1 (minoritarios)
# ===========================
df_antiguos_1 = df_antiguos_con_alerta[df_antiguos_con_alerta['target'] == 1]  # Alerta y target == 1
df_antiguos_0 = df_antiguos_con_alerta[df_antiguos_con_alerta['target'] == 0]  # Alerta y target == 0

# ===========================
# 4️⃣ Balanceo de clases en TRAIN
# ===========================
n_pos = len(df_antiguos_1)  # Cantidad de clases 1 (minoritarias)
n_neg_deseado = int(n_pos * (99.5 / 0.5))  # Queremos 99.5% de clase 0

# Ajustar clase 0 (target == 0)
if n_neg_deseado <= len(df_antiguos_0):
    df_antiguos_0_bal = resample(df_antiguos_0, replace=False, n_samples=n_neg_deseado, random_state=42)
else:
    df_antiguos_0_bal = resample(df_antiguos_0, replace=True, n_samples=n_neg_deseado, random_state=42)

# ===========================
# 5️⃣ Combinar con clases 1 (target == 1)
# ===========================
df_antiguos_1_bal = df_antiguos_1  # Mantener todas las clases 1 (no se ajustan, ya que se mantiene su porcentaje)

# Combinar 0 (balanceado) y 1 (original) para el dataset de entrenamiento
df_antiguos_balanceados = pd.concat([df_antiguos_0_bal, df_antiguos_1_bal], axis=0)

# ===========================
# 6️⃣ Tomamos el 0.5% de los SIN alerta
# ===========================
# Para tener el 99.5% de clase 0 y 0.5% de clase 1 en el TRAIN
porcentaje_sin_alerta = 0.005  # 0.5% de los registros sin alerta

# Tomamos el 0.5% de los casos sin alerta (sin alertas)
df_sin_alerta_sample = df_antiguos_sin_alerta.sample(frac=porcentaje_sin_alerta, random_state=42)

# ===========================
# 7️⃣ Concatenar alertas + 0.5% de sin alerta
# ===========================
df_train = pd.concat([df_antiguos_balanceados, df_sin_alerta_sample], axis=0)

# Barajamos los datos para evitar sesgo de orden
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

# ===========================
# 8️⃣ Mantener df_test intacto
# ===========================
# Filtrar df_test para asegurarnos de que tiene solo los registros del test (cod_mes >= 202508)
df_test=df_2[df_2['cod_mes'].between(202508, 202604)].copy()
#df_test = df_3[df_3['cod_mes'] >= 202508].copy()

# ===========================
# 9️⃣ Combinar df_train y df_test para df_7
# ===========================
df_3 = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# ===========================
# 10️⃣ Borrar columna 'tipo_alerta_n2' de df_7
# ===========================
#df_4 = df_4.drop(columns=['tipo_alerta_n2'], errors='ignore')

# ===========================
# 11️⃣ Verificar distribución final
# ===========================
print("Distribución final de clases en df_train (target):")
print(df_train['target'].value_counts(normalize=True))

print("\nDistribución final de tipo_alerta_n2 en df_train (Eliminada):")
print(df_train['tipo_alerta_n2'].value_counts())

print("\nDistribución final de clases en df_test (target):")
print(df_test['target'].value_counts(normalize=True))

print("\nDistribución final de clases en df_7 (target):")
print(df_3['target'].value_counts(normalize=True))

Distribución final de clases en df_train (target):
target
0   0.99
1   0.01
Name: proportion, dtype: float64

Distribución final de tipo_alerta_n2 en df_train (Eliminada):
tipo_alerta_n2
SIN_INFO           173993
AUTOMATICA            834
MANUAL                416
SEMI AUTOMATICA       157
Name: count, dtype: Int64

Distribución final de clases en df_test (target):
target
0   1.00
1   0.00
Name: proportion, dtype: float64

Distribución final de clases en df_7 (target):
target
0   1.00
1   0.00
Name: proportion, dtype: float64


In [18]:
df_3[['cod_mes', 'target']].value_counts()

cod_mes  target
202604   0         172271
202512   0         171315
202601   0         170951
202603   0         170300
202602   0         168998
202511   0         166252
202510   0         164651
202509   0         162739
202508   0         161296
202506   0          25963
202507   0          25908
202502   0          24662
202504   0          24632
202505   0          24596
202503   0          24436
202501   0          24326
202604   1            234
202602   1            150
202505   1            147
202502   1            145
202503   1            137
202603   1            126
202501   1            125
202510   1            123
202601   1            119
202508   1            119
202506   1            117
202504   1            109
202509   1             99
202507   1             97
202511   1             65
202512   1             16
Name: count, dtype: int64

In [19]:
# Eliminar columnas 'fec_constitucion' si existe
cols_a_eliminar = ["fec_constitucion"]
df_3 = df_3.drop(columns=cols_a_eliminar, errors="ignore")

In [20]:
df_3['tipo_alerta_n2'] = df_3['tipo_alerta_n2'].astype(str)

In [21]:
df_3["cnt_alerta_hist"] = (
    df_3["cnt_alerta_hist"]
    .astype("float64")
)

In [22]:
df_3["mto_fact_declarado_sunat"] = pd.to_numeric(df_3["mto_fact_declarado_sunat"], errors="coerce")

In [23]:
df_3.to_parquet(
    's3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/DATA_INFERENCIA/data_pn_total_expandido_new.parquet',
    index=False
)

In [24]:
df_3.shape

(1685224, 65)